# 🧹 Exercise 02 — Explore & Clean

**Day 1 · Guided**

Profile a DataFrame, fix dtypes, handle missing values, remove duplicates, and save a clean subset.

---

In [2]:
import pandas as pd
import geopandas as gpd

# TODO: Replace with your city slug (same as Exercise 01)
CITY_SLUG = "종로구"  # <-- change this!

# Load from saved GeoPackage
gdf = gpd.read_file(f"../data/raw_{CITY_SLUG}.gpkg")

# Extract lat/lon from geometry and convert to regular DataFrame
gdf["lat"] = gdf.geometry.centroid.y
gdf["lon"] = gdf.geometry.centroid.x
df = pd.DataFrame(gdf.drop(columns=["geometry"]))

# Rename osmid → id, drop osmnx metadata
if "osmid" in df.columns:
    df = df.rename(columns={"osmid": "id"})
for col in ["nodes", "ways"]:
    if col in df.columns:
        df = df.drop(columns=[col])

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")

Loaded 1627 rows, 173 columns


C:\Users\maria\AppData\Local\Temp\ipykernel_29732\2578219686.py:11: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lat"] = gdf.geometry.centroid.y
C:\Users\maria\AppData\Local\Temp\ipykernel_29732\2578219686.py:12: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["lon"] = gdf.geometry.centroid.x


### 2.1 📊 Profile the data

Before fixing anything, understand what you have. Three best friends: `.info()`, `.describe()`, `.dtypes`.

In [3]:
# .info() gives you: columns, non-null counts, dtypes, memory usage
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1627 entries, 0 to 1626
Columns: 173 entries, element to lon
dtypes: float64(2), int64(1), str(170)
memory usage: 2.1 MB


In [4]:
# TODO: Run .describe() — what does it tell you about lat/lon?
# Does the range make sense for your city?

In [5]:
# TODO: Check the dtypes of lat and lon.
# Are they float64? If not, that's a problem we need to fix.
# Hint: df["lat"].dtype

**Think:** How many columns? What % non-null? Are `lat`/`lon` float64? Does the coordinate range match your district?

### 2.2 🔧 Fix data types

Coordinates must be numeric. In some exports they arrive as strings.

In [6]:
# Ensure lat/lon are float — pd.to_numeric handles strings → float
# errors="coerce" turns unparseable values into NaN instead of crashing
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

print(f"lat dtype: {df['lat'].dtype}")
print(f"lon dtype: {df['lon'].dtype}")

# Drop rows where conversion failed
before = len(df)
df = df.dropna(subset=["lat", "lon"])
print(f"Dropped {before - len(df)} rows with invalid coordinates")

lat dtype: float64
lon dtype: float64
Dropped 0 rows with invalid coordinates


### 2.3 🕳️ Handle missing values

Most tag columns are >90% empty — normal for OSM, not every node has every tag.

In [7]:
# Calculate fill rate for each column
fill_rate = (df.notna().sum() / len(df) * 100).sort_values(ascending=False)

print("Top 20 columns by fill rate:")
print(fill_rate.head(20).round(1).to_string())

print(f"\nColumns with <1% fill rate: {(fill_rate < 1).sum()}")
print(f"Columns with <5% fill rate: {(fill_rate < 5).sum()}")

Top 20 columns by fill rate:
element             100.0
id                  100.0
amenity             100.0
lon                 100.0
lat                 100.0
name                 88.6
addr:city            35.7
addr:district        35.5
addr:housenumber     31.4
addr:street          31.4
name:en              30.6
addr:postcode        29.8
addr:subdistrict     29.4
cuisine              25.7
name:ko              17.3
phone                11.9
building              9.6
name:ja               8.0
website               7.6
brand                 6.3

Columns with <1% fill rate: 119
Columns with <5% fill rate: 149


In [8]:
# TODO: Drop columns with very low fill rate (<1%)
# But keep important columns even if sparse: cuisine, opening_hours, phone, website, wheelchair

# Hint:
# sparse_cols = fill_rate[fill_rate < 1].index.tolist()
# keep_anyway = {"cuisine", "opening_hours", "phone", "website", "wheelchair"}
# drop_cols = [c for c in sparse_cols if c not in keep_anyway]
# df = df.drop(columns=drop_cols)

### 2.4 🔁 Remove duplicates

Same coordinates but different IDs — happens from mass imports or editing mistakes.

In [9]:
# TODO: How many duplicate coordinates are there?
# Hint: df.duplicated(subset=["lat", "lon"], keep=False).sum()

In [10]:
# TODO: Look at some duplicates — are they truly the same place or different things at the same spot?
# Hint:
# dupes = df[df.duplicated(subset=["lat", "lon"], keep=False)]
# dupes.sort_values(["lat", "lon"]).head(10)[["id", "lat", "lon", "amenity", "name"]]

In [11]:
# TODO: Remove duplicates (keep first occurrence)
# Hint:
# before = len(df)
# df = df.drop_duplicates(subset=["lat", "lon"], keep="first")
# print(f"Removed {before - len(df)} duplicates")

**Think:** We used `keep="first"` — is that always right? What if the second duplicate has better tag data?

### 2.5 🏷️ Extract a primary name

OSM has `name`, `name:en`, `name:sv`, `name:ar`… We need one usable column.

In [12]:
# What name columns do we have?
name_cols = [c for c in df.columns if c.startswith("name")]
print(f"Name-related columns: {sorted(name_cols)}")

Name-related columns: ['name', 'name:de', 'name:en', 'name:fr', 'name:hi', 'name:ja', 'name:ko', 'name:ko-Hani', 'name:ko-Latn', 'name:pa', 'name:pnb', 'name:ru', 'name:ur', 'name:zh', 'name:zh-Hans', 'name:zh-Hant']


In [21]:
# Create a 'primary_name' column
# Priority: name > name:en > first available name:* column

def get_primary_name(row):
    # Check name column first
    if pd.notna(row.get("name")):
        return row["name"]
    # Then check name:en
    if pd.notna(row.get("name:en")):
        return row["name:en"]
    # Then check any other name:* columns
    for col in row.index:
        if col.startswith("name:") and pd.notna(row[col]):
            return row[col]
    # Return None if no name found
    return None

df["primary_name"] = df.apply(get_primary_name, axis=1)
print(f"primary_name column created: {df['primary_name'].notna().sum()} amenities have names")

primary_name column created: 1578 amenities have names


### 2.6 💾 Select core columns & save

In [22]:
core_cols = ["id", "lat", "lon", "amenity", "primary_name", "cuisine", "opening_hours"]
optional_cols = ["phone", "website", "wheelchair", "addr:street", "addr:housenumber"]

# Only keep columns that exist in df
available_core = [c for c in core_cols if c in df.columns]
available_optional = [c for c in optional_cols if c in df.columns]

print(f"Core columns: {available_core}")
print(f"Optional columns: {available_optional}")

df_clean = df[available_core + available_optional].copy()
print(f"\nClean DataFrame shape: {df_clean.shape}")
print(f"Columns: {list(df_clean.columns)}")

Core columns: ['id', 'lat', 'lon', 'amenity', 'primary_name', 'cuisine', 'opening_hours']
Optional columns: ['phone', 'website', 'wheelchair', 'addr:street', 'addr:housenumber']

Clean DataFrame shape: (1627, 12)
Columns: ['id', 'lat', 'lon', 'amenity', 'primary_name', 'cuisine', 'opening_hours', 'phone', 'website', 'wheelchair', 'addr:street', 'addr:housenumber']


In [23]:
df_clean.to_csv(f"../data/clean_{CITY_SLUG}.csv", index=False)
print(f"✅ Saved clean DataFrame to ../data/clean_{CITY_SLUG}.csv")

✅ Saved clean DataFrame to ../data/clean_종로구.csv


---

✅ Correct dtypes · ✅ No duplicate coordinates · ✅ Sparse columns removed · ✅ `primary_name` column · ✅ Clean CSV saved

#### Checklist
- [ ] `df_clean.dtypes` → `lat`/`lon` are `float64`
- [ ] `df_clean.duplicated(subset=['lat', 'lon']).sum()` → `0`
- [ ] `df_clean['primary_name'].notna().mean()` → > 0.5

**Next →** [Exercise 03 — Tag Normalization](03_tag_normalization.ipynb)